# Calibration on the bundled sample data

This notebook runs both calibrations on the small real fixtures committed under
`examples/data/` (see `examples/data/README.md`), so it needs **no external archive**.

- **CHM15k (1064 nm)** and **Mini-MPL (532 nm)** Rayleigh run fully offline.
- **CL61 / CL31 / CL51 (910 nm)** need a monthly **CAMS** file for the mandatory
  water-vapour correction (the WV LUT itself is bundled). Those cells are guarded and
  print a notice if CAMS is not on this machine.

## 1. Setup

In [ ]:
import tempfile
from pathlib import Path
from netCDF4 import Dataset

from calibration import calibrate_rayleigh, CalibrationOptions, InstrumentInfo
from calibration.config import InstrumentType, DataLevel
from calibration.cloud import liquid_cloud_calibration, CloudCalConfig
from calibration.water_vapor_correction.water_vapor import DEFAULT_ABS_CROSS_SECTION

# locate the repo whether the notebook runs from the repo root or from examples/
ROOT = Path.cwd()
if (ROOT / "examples" / "data").exists():
    pass
elif (ROOT.parent / "examples" / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"
OPTIONS = ROOT / "options.json"

CAMS = Path(CalibrationOptions.from_json(OPTIONS).cams_folder)
print("sample data :", DATA)
print("CAMS present:", CAMS.exists(), "->", CAMS, "(needed only for 910 nm)")

def l2(wmo, date):
    return DATA / "L2" / wmo / "2026" / date[4:6] / f"L2_{wmo}_A{date}.nc"

def info_from(wmo, itype, ncpath):
    with Dataset(ncpath) as ds:
        lat = float(ds.variables["station_latitude"][...])
        lon = float(ds.variables["station_longitude"][...])
        alt = float(ds.variables["station_altitude"][...])
    return InstrumentInfo(site_name=wmo, wmo_id=wmo, identifier="A",
                          instrument_type=itype, latitude=lat, longitude=lon, altitude=alt)

def rayleigh_options():
    o = CalibrationOptions.from_json(OPTIONS)
    o.data_level = DataLevel.L2_DAILY
    o.folder_root = DATA / "L2"
    o.cams_folder = CAMS
    o.abs_cs_lookup_table = Path("")          # use the bundled WV LUT
    o.folder_output = Path(tempfile.mkdtemp())
    for k in ("plot_main", "plot_all"):
        if hasattr(o, k):
            setattr(o, k, 0)
    return o

## 2. Rayleigh calibration — CHM15k (1064 nm), self-contained

A clear night at Payerne. 1064 nm is insensitive to water vapour, so this runs with no
external data at all.

In [ ]:
res = calibrate_rayleigh("20260225", info_from("0-20000-0-06610", InstrumentType.CHM15k,
                         l2("0-20000-0-06610", "20260225")), rayleigh_options())
print("CHM15k  flag =", res.flag, "(", res.flag_meaning, ")   lidar constant =", f"{res.lidar_constant:.3e}")

### Mini-MPL (532 nm), self-contained — a clear night at Lille

In [ ]:
res = calibrate_rayleigh("20260423", info_from("0-20000-0-07014", InstrumentType.MINI_MPL,
                         l2("0-20000-0-07014", "20260423")), rayleigh_options())
print("Mini-MPL  flag =", res.flag, "   lidar constant =", f"{res.lidar_constant:.3e}")

## 3. CL61 (910 nm) — Rayleigh **and** cloud

CL61 is a 910 nm instrument, so both calibrations apply the water-vapour correction
(bundled LUT + monthly CAMS). The cells run only if CAMS is present.

In [ ]:
if not CAMS.exists():
    print("CAMS not found - skipping the 910 nm CL61 examples. Set cams_folder in options.json.")
else:
    # 3a. Rayleigh on a clear CL61 night (Sion)
    res = calibrate_rayleigh("20260304", info_from("0-756-4-EERLCL61", InstrumentType.CL61,
                             l2("0-756-4-EERLCL61", "20260304")), rayleigh_options())
    print("CL61 Rayleigh  flag =", res.flag, "   lidar constant =", f"{res.lidar_constant:.3e}")

    # 3b. Liquid-cloud calibration on an overcast CL61 day
    cfg = CloudCalConfig(
        nc_file=str(l2("0-756-4-EERLCL61", "20260414")),
        cams_folder=str(CAMS),
        abs_cs_lookup_table=str(DEFAULT_ABS_CROSS_SECTION),   # bundled WV LUT
        apply_wv_correction=1, aerosol_lidar_ratio=50,
    )
    cloud = liquid_cloud_calibration(cfg)
    print("CL61 cloud     in-cloud profiles =", cloud.n_profiles,
          "   cal_median =", cloud.cal_median)

### Cloud calibration that yields a constant — CL31 / CL51 (overcast)

In [ ]:
if not CAMS.exists():
    print("CAMS not found - skipping the 910 nm cloud examples.")
else:
    for wmo, date in [("0-20000-0-06602", "20260220"), ("0-20000-0-02998", "20260116")]:
        cfg = CloudCalConfig(nc_file=str(l2(wmo, date)), cams_folder=str(CAMS),
                             abs_cs_lookup_table=str(DEFAULT_ABS_CROSS_SECTION),
                             apply_wv_correction=1, aerosol_lidar_ratio=50)
        r = liquid_cloud_calibration(cfg)
        print(f"{wmo}  n={r.n_profiles:4d}  calibration coefficient C = {r.cal_median:.3f}")

See `tests/test_sample_data.py` for these same fixtures wired as automated tests,
and `examples/data/README.md` for the full fixture list and how each day was chosen.